In [1]:
import gc
import os
import sys
from pathlib import Path
import jax
jax.config.update("jax_traceback_filtering", "off")
jax.config.update("jax_compilation_cache_dir", "../jax-caches")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
import numpy as np
import jax.numpy as jnp
from scipy.optimize import minimize, least_squares
from desc.dipole import _Dipole, DipoleSet, import_dipoles, export_dipoles
from desc.io import load
from desc.grid import LinearGrid, Grid
from desc.magnetic_fields._core import dipole_field
from desc.objectives import (ObjectiveFunction, DipoleDiscreteness, DipoleVolume, QuadraticFluxPM )
from desc.optimize import Optimizer
import matplotlib.pyplot as plt
from desc.magnetic_fields import SumMagneticField, SplineMagneticField
from desc.compute import get_params
from desc.equilibrium import Equilibrium
from desc.plotting import plot_comparison, plot_section, plot_2d, plot_3d, plot_coils
from desc.coils import CoilSet, FourierPlanarCoil
from desc.integrals import compute_B_plasma
from desc.coils import initialize_modular_coils

eq = load('input_output_files/NFP2_s12_A10_mr0.3_eq_final_scaled_R0_p5m.h5')
coilset_qi = load("qipm_coils.h5")

In [ ]:
# generate circular planar coils
a = eq.compute("a")["a"]
num_coils=8
coilset = initialize_modular_coils(eq, num_coils=num_coils, r_over_a=0.16 / a)
plot_grid = LinearGrid(
    rho=1.0,
    M=40,
    N=80,
    NFP=1,
    sym=False,
    endpoint=True,
)
coil_grid = LinearGrid(N=40)
calc_grid = LinearGrid(rho=[0.0], M=0, zeta=np.linspace(0, np.pi / eq.NFP, num_coils, 
    endpoint=False) + np.pi / (2 * eq.NFP * num_coils), NFP=eq.NFP)

# fig = plot_3d(
#     eq.surface, "B*n", field=coilset, field_grid=coil_grid, grid=plot_grid
# )
# fig = plot_coils(coilset, fig=fig)
# fig.show()

axis_data = eq.compute(
    ["|B|", "X", "Y", "Z"],
    grid=calc_grid,
)

axis_pts = np.column_stack([
    np.asarray(axis_data["X"]).ravel(),
    np.asarray(axis_data["Y"]).ravel(),
    np.asarray(axis_data["Z"]).ravel(),
])

B_eq = np.asarray(axis_data["|B|"]).ravel()

I0 = np.array([float(c.current) for c in coilset])

def residual(I):
    for c, current in zip(coilset, I):
        c.current = current

    B_vec = coilset.compute_magnetic_field(
        axis_pts,
        basis="xyz",
    )

    B_coils = np.linalg.norm(np.asarray(B_vec), axis=-1)

    return B_coils - B_eq

rms_initial = np.sqrt(np.mean(residual(I0)**2))

result = least_squares(
    residual,
    I0,
    method="trf",
    x_scale="jac",
    max_nfev=2000,
    verbose=1,
)

I_opt = result.x

for c, current in zip(coilset, I_opt):
    c.current = current

r_final = residual(I_opt)
rms_final = np.sqrt(np.mean(r_final**2))

print("\nOptimization success:", result.success)
print("Message:", result.message)
print(f"Initial RMS error: {rms_initial:.6e} T")
print(f"Final RMS error:   {rms_final:.6e} T")
print("Optimized currents [A]:", I_opt)

B_coils_opt = np.linalg.norm(
    np.asarray(
        coilset.compute_magnetic_field(axis_pts, basis="xyz")
    ),
    axis=-1,
)
plot_grid = LinearGrid(M=20, N=40, NFP=1, endpoint=True)
axis_grid = LinearGrid(rho=0, theta = 0, N=80, NFP=1, endpoint=True)
fig, ax = plt.subplots(1,1, figsize = (10,6))
data_axis_eq = eq.compute(['|B|','X','Y','Z'], grid = axis_grid)

axis_pts = np.array([data_axis_eq["X"],data_axis_eq["Y"],data_axis_eq["Z"]]).T

ax.plot(data_axis_eq['|B|'], label='|B| eq')
ax.set_title('|B| on axis: Coils vs Eq')

modB_on_axis_coils = np.linalg.norm(coilset.compute_magnetic_field(axis_pts, basis='xyz'), axis = -1)
ax.plot(modB_on_axis_coils,label='|B| Coils')

ax.legend()

In [ ]:
from desc.dipole import initialize_dipole_grid

qi_pm_grid = initialize_dipole_grid(
    eq=eq,
    target_size=0.25 * 0.0254,
    L=14,
    layer_thick=(1/16) * 0.0254,
    wall_thick=0.001,
    d_offset=0.02,
    gap=0.0,
    rho=1.0,
    M_surface=96,
    N_surface=128,
    verbose=True,
)

print(f"Unique dipoles: {len(qi_pm_grid):,}")

In [ ]:
def calculate_bn_error(
    eq,
    dipoles,
    coils,
    M=64,
    N=64,
    chunk_size=8,
    plot=True,
):
    """
    Calculate normalized normal magnetic field error on the LCFS.

    Parameters
    ----------
    eq : Equilibrium
        Equilibrium defining the plasma boundary.
    dipoles : DipoleSet
        Permanent magnets producing the dipole field.
    coils : CoilSet
        Coils producing the background magnetic field.
    M, N : int
        Poloidal and toroidal grid resolutions.
    chunk_size : int
        Chunk size for magnetic field calculations.
    plot : bool
        Whether to plot normalized Bn error.

    Returns
    -------
    results : dict
        Normalized Bn error, field values, and summary statistics.
    """

    # Surface grid
    grid = LinearGrid(
        rho=1.0,
        M=M,
        N=N,
        NFP=eq.NFP,
        sym=False,
        endpoint=False,
    )

    # Plasma surface coordinates and unit normals (Cartesian)
    data = eq.compute(["x", "n_rho"], grid=grid, basis="xyz")

    xyz = np.asarray(data["x"])
    normals = np.asarray(data["n_rho"])
    normals = normals/(np.linalg.norm(normals, axis=1, keepdims=True))

    # Coil magnetic field
    B_coils = np.asarray(
        coils.compute_magnetic_field(
            xyz,
            basis="xyz",
            chunk_size=chunk_size,
        )
    )

    # Permanent magnet magnetic field
    B_pm = np.asarray(
        dipoles.compute_magnetic_field(
            xyz,
            basis="xyz",
            chunk_size=chunk_size,
        )
    )

    # Total normal magnetic field
    B_total = B_coils + B_pm
    Bn = np.sum(B_total * normals, axis=1)

    # Local normalization by coil magnetic field magnitude
    B_coils_mag = np.linalg.norm(B_coils, axis=1)

    if np.any(B_coils_mag == 0):
        raise ValueError("Coil magnetic field is zero at some surface points.")

    normalized_Bn = Bn / B_coils_mag

    # Summary statistics
    mean_error = np.mean(np.abs(normalized_Bn))
    rms_error = np.sqrt(np.mean(normalized_Bn**2))
    max_error = np.max(np.abs(normalized_Bn))

    print("========== NORMALIZED BN ERROR ==========")
    print(f"Mean |Bn/Bcoils|: {mean_error:.6%}")
    print(f"RMS  |Bn/Bcoils|: {rms_error:.6%}")
    print(f"Max  |Bn/Bcoils|: {max_error:.6%}")

    # Reshape using grid node coordinates
    nodes = np.asarray(grid.nodes)

    theta, i = np.unique(nodes[:, 1], return_inverse=True)
    phi, j = np.unique(nodes[:, 2], return_inverse=True)

    error_2d = np.empty((len(theta), len(phi)))
    error_2d[i, j] = normalized_Bn

    if plot:
        vmax = np.max(np.abs(error_2d))
        levels = np.linspace(-vmax, vmax, 51)

        fig, ax = plt.subplots(figsize=(10, 6))

        cf = ax.contourf(
            phi,
            theta,
            error_2d,
            levels=levels,
            cmap="RdBu_r",
            extend="both",
        )

        cbar = fig.colorbar(cf, ax=ax, pad=0.03)
        cbar.set_label(
            r"$\mathbf{B}\cdot\hat{\mathbf{n}}/|\mathbf{B}_{\rm coils}|$",
            fontsize=22,
        )

        formatter = ScalarFormatter(useMathText=True)
        formatter.set_scientific(True)
        formatter.set_powerlimits((0, 0))

        cbar.formatter = formatter
        cbar.update_ticks()
        cbar.ax.tick_params(labelsize=15)
        cbar.ax.yaxis.get_offset_text().set_fontsize(14)

        ax.set_xlabel(r"Toroidal angle $\phi$", fontsize=20)
        ax.set_ylabel(r"Poloidal angle $\theta$", fontsize=20)
        ax.tick_params(labelsize=15)

        ax.set_xlim(phi.min(), phi.max())
        ax.set_ylim(0, 2 * np.pi)

        fig.tight_layout()
        plt.show()

    return {
        "grid": grid,
        "Bn": Bn,
        "normalized_Bn": normalized_Bn,
        "error_2d": error_2d,
        "B_coils": B_coils,
        "B_pm": B_pm,
        "mean_error": mean_error,
        "rms_error": rms_error,
        "max_error": max_error,
    }


def calculate_dpbin(dipoles):
    """Calculate DipoleDiscreteness's raw residual and squared penalty."""
    rho = np.asarray(dipoles.rho_tilde, dtype=float)

    d = np.abs(rho) * (1.0 - np.abs(rho))

    dpbin = np.sqrt(np.sum(d**2))

    print(f"Discreteness residual (dpbin): {dpbin:.6e}")
    print(f"Discreteness sum of squares:   {dpbin**2:.6e}")

    return dpbin


def calculate_dpvol(dipoles, eps=1e-6):
    """Calculate DipoleVolume's raw residuals and squared penalty."""
    rho = np.asarray(dipoles.rho_tilde, dtype=float)

    v = (rho**2 + eps**2)**0.25

    dpvol = np.sum(v**2)

    print(f"Dipole volume penalty (dpvol): {dpvol:.6e}")
    print(f"Approximate sum |rho_tilde|:   {np.sum(np.abs(rho)):.6e}")

    return dpvol

In [ ]:
from desc.plotting import plot_3d, plot_coils, plot_dipoles
# plot MUSE solution

grid = LinearGrid(
    rho=1.0,
    M=40,
    N=80,
    NFP=1,
    sym=False,
    endpoint=True,
)

fig = plot_3d(eq, "|B|", grid=grid)

# Remove surface colorbar
fig.update_traces(
    showscale=False,
    selector=dict(type="surface"),
)

fig = plot_coils(
    coilset_qi,
    fig=fig,
    color="#510076",
    lw=7,
    showgrid=False,
    showticklabels=False,
    showaxislabels=False,
    zeroline=False,
)

fig = plot_dipoles(
    qi_pm_grid,
    fig=fig,
    threshold=0.5,
    size=2,
    opacity=0.85,
    color_positive="#D20000",
    color_negative="#004BE2",
    showgrid=False,
    showticklabels=False,
    showaxislabels=False,
    zeroline=False,
)

fig.update_layout(
    width=1200,
    height=1000,
    margin=dict(l=0, r=0, t=0, b=0),
    paper_bgcolor="white",
    showlegend=False,
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=0.8),
        ),
    ),
)

fig.show()

In [ ]:
qi_pm_grid.rho=np.zeros(len(qi_pm_grid))

In [ ]:
qi_pm_grid[100].rho

In [ ]:
coil_grid   = LinearGrid(N=40)
plasma_grid = LinearGrid(M=15, N=30, NFP=eq.NFP, sym=eq.sym)

weights = {"quadratic flux": 1000,"discreteness": 1,"volume": 5}

optimizer = Optimizer("scipy-l-bfgs-b")
qflux = QuadraticFluxPM(
    eq,
    field=qi_pm_grid,
    eval_grid=plasma_grid,
    field_fixed=coilset_qi,
    vacuum=True,
    weight=weights["quadratic flux"],
    bs_chunk_size=5,
    jac_chunk_size=10,
)
obj = ObjectiveFunction(
    (qflux,),
    deriv_mode="blocked",
            # DipoleDiscreteness(
            #     dipole=dipoles, weight=weights["discreteness"], 
            # ),
            # DipoleVolume(
            #     dipole=dipoles, weight=weights["volume"], 
            # )
        
     )

(qi_continuous_opt,), _ = optimizer.optimize(
    qi_pm_grid,
    objective=obj,
    maxiter=500,
    verbose=3,
    ftol=1e-8,
    copy=True,
)

In [ ]:
from matplotlib.ticker import ScalarFormatter
results = calculate_bn_error(
    eq=eq,
    dipoles=qi_continuous_opt,
    coils=coilset_qi,
    M=32,
    N=32,
    plot=True,
)

In [ ]:
# plot rho distribution

import numpy as np
import matplotlib.pyplot as plt

def plot_rho_distribution(dipoles, name="MUSE", bins=80):
    """
    Plot the distribution of normalized dipole strengths.

    Parameters
    ----------
    dipoles : DipoleSet
        Dipole configuration to analyze.
    name : str
        Configuration name for labeling and saving.
    bins : int
        Number of histogram bins.
    """
    rho = np.asarray(dipoles.rho_tilde)

    print(f"========== {name} ==========")
    print(f"Number of dipoles: {len(rho):,}")
    print(f"Mean |rho_tilde|:  {np.mean(np.abs(rho)):.6f}")
    print(f"L2 norm:           {np.linalg.norm(rho):.6f}")
    print(f"Near zero (<0.05): {np.mean(np.abs(rho) < 0.05):.2%}")
    print(f"Near ±1 (>0.95):   {np.mean(np.abs(rho) > 0.95):.2%}")
    print(f"Active (>0.5):     {np.mean(np.abs(rho) > 0.5):.2%}")

    fig, ax = plt.subplots(figsize=(9, 5.5))

    ax.hist(
        rho,
        bins=np.linspace(-1, 1, bins + 1),
        color="tab:blue" if name == "QI" else "tab:red",
        edgecolor="white",
        linewidth=0.3,
        alpha=0.85,
    )

    ax.set_xlabel(r"Normalized dipole strength $\tilde{\rho}$", fontsize=20)
    ax.set_ylabel("Number of dipoles", fontsize=20)
    ax.tick_params(labelsize=16)
    ax.set_xlim(-1, 1)

    fig.tight_layout()
    plt.show()

    return rho

rho_muse = plot_rho_distribution(
    qi_continuous_opt,
    name="MUSE",
)